In [1]:
import datetime
import logging
import sqlite3
import uuid
import bet_placing
import market_resolution
import importlib
importlib.reload(market_resolution)
importlib.reload(bet_placing)

<module 'bet_placing' from '/Users/prathamwankhede/Documents/predict/bet_placing.py'>

In [2]:
conn = sqlite3.connect('bsname.db')
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

In [5]:
def create_user(db: sqlite3.connect, username: str, initial_balance: float = 500.0, user_id: str = str(uuid.uuid4())) -> bool:
    db.row_factory = sqlite3.Row
    cursor = db.cursor()
    cursor.execute("""
        INSERT INTO users (id, username, balance, created_at)
        VALUES (?, ?, ?, ?);
    """, (user_id, username, initial_balance, datetime.datetime.now()))
    db.commit()
    logging.info(f"Created user {username} with id {user_id} and initial balance {initial_balance}")
    return user_id

In [60]:
cursor.execute("""
    DELETE FROM users
    Where username = 'Nelson Bighetti';
""")
conn.commit()

In [50]:
uid = create_user(conn, "Nelson Bighetti")

/var/folders/s9/7x6wjtq126773lhfsl5rqqv40000gn/T/ipykernel_6534/545949362.py:4: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("""
2026-03-21 04:22:11,218 INFO Created user Nelson Bighetti with id 5b5099ba-9f4e-4471-83a9-e2f3bb092912 and initial balance 500.0


In [3]:
market_id = bet_placing.create_market(conn, "Will Big Head become the president of Stanford?", datetime.datetime.now(), datetime.datetime.now() + datetime.timedelta(days=1), 1, 1)

2026-05-20 01:39:23,033 INFO Created market 0c2b78a8-1a28-4a9f-89fc-86689e499890 with start time 2026-05-20 01:39:23.024177, end time 2026-05-21 01:39:23.024562, c_yes 1 and c_no 1


In [59]:
#7ba57abb-6c3a-45ae-b1ff-2fa149a099d4
#34050627-ae6d-4926-b667-6dbba548c323
cursor.execute("""
    DELETE FROM markets
    Where name = 'Will Big Head become the president of Stanford?';
""")
conn.commit()

In [52]:
bet_placing.placeBet(conn, market_id, 1.0, datetime.datetime.now(), 100, 1, uid)

2026-03-21 04:22:19,990 INFO User 5b5099ba-9f4e-4471-83a9-e2f3bb092912 executed trade of 100 shares at price 1.0 on side yes for market 234414f4-785b-47e4-8179-d733cddc93b7 at time 2026-03-21 04:22:19.988173 with weight 2.9999305782511803


In [4]:
market_resolution.resolve_market(conn, market_id, "yes")

2026-05-20 01:39:44,018 INFO Resolved market 0c2b78a8-1a28-4a9f-89fc-86689e499890 with resolution yes


In [58]:
market_resolution.distribute_winnings(conn, market_id)

2026-03-21 04:24:08,621 INFO Distributed 100.0 to user 5b5099ba-9f4e-4471-83a9-e2f3bb092912 for market 234414f4-785b-47e4-8179-d733cddc93b7


In [48]:
uid = 'f5f56be4-3eef-48e6-a497-61746869e7fd'
cursor.execute("""
    Select * from users;
""")
row = cursor.fetchall()
print(dict(row) if row else "No user found")

ValueError: dictionary update sequence element #0 has length 4; 2 is required

## Isolated Payout Test
This test creates a temporary user and market, places a single bet, resolves the market, runs distribution, and prints full diagnostics.
It also cleans up inserted rows at the end so your main data remains unchanged.

In [6]:
import uuid

def run_isolated_payout_test(db: sqlite3.Connection, resolve_side: str = "yes", bet_yes: bool = True):
    db.row_factory = sqlite3.Row
    cur = db.cursor()

    test_user_id = str(uuid.uuid4())
    test_username = f"tmp_test_{test_user_id[:8]}"
    test_market_name = f"tmp_market_{test_user_id[:8]}"

    print("--- Setup ---")
    cur.execute(
        """
        INSERT INTO users (id, username, balance, created_at)
        VALUES (?, ?, ?, ?)
        """,
        (test_user_id, test_username, 500.0, datetime.datetime.now()),
    )
    db.commit()

    test_market_id = bet_placing.create_market(
        db,
        test_market_name,
        datetime.datetime.now(),
        datetime.datetime.now() + datetime.timedelta(days=1),
        1,
        1,
    )

    bet_placing.placeBet(
        db,
        test_market_id,
        1.0,
        datetime.datetime.now(),
        100,
        bet_yes,
        test_user_id,
    )

    cur.execute("SELECT balance FROM users WHERE id = ?", (test_user_id,))
    balance_after_bet = cur.fetchone()["balance"]
    print("Balance after placing bet:", balance_after_bet)

    cur.execute(
        """
        SELECT side, shares, price, weighted_bet
        FROM bets
        WHERE market_id = ?
        """,
        (test_market_id,),
    )
    print("Bet row:", dict(cur.fetchone()))

    market_resolution.resolve_market(db, test_market_id, resolve_side)

    cur.execute(
        """
        SELECT status, resolution_side, tot_pool, weighted_pool_yes, weighted_pool_no
        FROM markets
        WHERE id = ?
        """,
        (test_market_id,),
    )
    print("Market row before distribution:", dict(cur.fetchone()))

    print("--- Distribute ---")
    market_resolution.distribute_winnings(db, test_market_id)

    cur.execute("SELECT balance FROM users WHERE id = ?", (test_user_id,))
    final_balance = cur.fetchone()["balance"]
    print("Final balance after distribution:", final_balance)

    winner = (resolve_side == "yes" and bet_yes) or (resolve_side == "no" and not bet_yes)
    expected = 500.0 if winner else 400.0
    print("Expected balance in single-bettor scenario:", expected)

    print("--- Cleanup ---")
    cur.execute("DELETE FROM bets WHERE market_id = ?", (test_market_id,))
    cur.execute("DELETE FROM markets WHERE id = ?", (test_market_id,))
    cur.execute("DELETE FROM users WHERE id = ?", (test_user_id,))
    db.commit()
    print("Cleanup complete")

# Run both cases:
print("\nCase A: Bet YES, resolve YES (winner expected)")
run_isolated_payout_test(conn, resolve_side="yes", bet_yes=True)

print("\nCase B: Bet YES, resolve NO (loser expected)")
run_isolated_payout_test(conn, resolve_side="no", bet_yes=True)

/var/folders/s9/7x6wjtq126773lhfsl5rqqv40000gn/T/ipykernel_61517/1657437268.py:12: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cur.execute(
2026-04-20 23:11:43,176 INFO Created market e3c40e75-d4c6-4808-85e2-d5c2bd243617 with start time 2026-04-20 23:11:43.174986, end time 2026-04-21 23:11:43.174995, c_yes 1 and c_no 1
2026-04-20 23:11:43,179 INFO User 76c96464-216f-47b0-8d3b-549b7b3ffbc4 executed trade of 100 shares at price 1.0 on side yes for market e3c40e75-d4c6-4808-85e2-d5c2bd243617 at time 2026-04-20 23:11:43.177873 with weight 2.999999966585648
2026-04-20 23:11:43,180 INFO Resolved market e3c40e75-d4c6-4808-85e2-d5c2bd243617 with resolution yes
2026-04-20 23:11:43,182 INFO Distributed 100.0 to user 76c96464-216f-47b0-8d3b-549b7b3ffbc4 for market e3c40e75-d4c6-4808-85e2-d5c2bd243617
2026-04-20 23:11:43,187 INFO Created market 298c35b6-3132-4ee9-bf94-cd2b5e83ec4e with start tim


Case A: Bet YES, resolve YES (winner expected)
--- Setup ---
Balance after placing bet: 400.0
Bet row: {'side': 'yes', 'shares': 100, 'price': 1.0, 'weighted_bet': 299.9999966585648}
Market row before distribution: {'status': 'resolved', 'resolution_side': 'yes', 'tot_pool': 100.0, 'weighted_pool_yes': 299.9999966585648, 'weighted_pool_no': 0.0}
--- Distribute ---
Final balance after distribution: 500.0
Expected balance in single-bettor scenario: 500.0
--- Cleanup ---
Cleanup complete

Case B: Bet YES, resolve NO (loser expected)
--- Setup ---
Balance after placing bet: 400.0
Bet row: {'side': 'yes', 'shares': 100, 'price': 1.0, 'weighted_bet': 299.99999770138885}
Market row before distribution: {'status': 'resolved', 'resolution_side': 'no', 'tot_pool': 100.0, 'weighted_pool_yes': 299.99999770138885, 'weighted_pool_no': 0.0}
--- Distribute ---
Final balance after distribution: 400.0
Expected balance in single-bettor scenario: 400.0
--- Cleanup ---
Cleanup complete


In [ ]:
import sqlite3
import pandas as pd
import datetime
from bet_placing import placeBet
from market_resolution import distribute_winnings

def run_multi_user_weight_test():
    db = sqlite3.connect('bsname.db')
    db.row_factory = sqlite3.Row
    cursor = db.cursor()
    
    # Create test users and give them initial balances
    user_ids = ['test_user_1', 'test_user_2', 'test_user_3']
    for uid in user_ids:
        cursor.execute("INSERT INTO users (id, username, balance, created_at) VALUES (?, ?, ?, ?)",
                       (uid, uid, 1000.0, datetime.datetime.now()))

    # Create a test market
    market_id = 'test_market_weight_123'
    start_time = datetime.datetime.now() - datetime.timedelta(seconds=10000)
    end_time = datetime.datetime.now() + datetime.timedelta(days=1)
    cursor.execute("INSERT INTO markets (id, name, started_at, ended_at, estimated_resolution_time) VALUES (?, ?, ?, ?, ?)",
                   (market_id, 'weight_test', start_time, end_time, end_time))
    db.commit()

    print("Initial Balances:")
    for uid in user_ids:
        b = cursor.execute("SELECT balance FROM users WHERE id = ?", (uid,)).fetchone()['balance']
        print(f"{uid}: {b}")
    print("-" * 30)

    # 1. user1 bets YES early
    placeBet(db, market_id, 0.5, datetime.datetime.now() - datetime.timedelta(seconds=8000), 100, True, 'test_user_1')
    
    # 2. user2 bets YES late
    placeBet(db, market_id, 0.5, datetime.datetime.now() - datetime.timedelta(seconds=1000), 100, True, 'test_user_2')
    
    # 3. user3 bets NO (providing the loser pool)
    placeBet(db, market_id, 0.5, datetime.datetime.now() - datetime.timedelta(seconds=500), 200, False, 'test_user_3')

    # Resolve market as YES
    cursor.execute("UPDATE markets SET status = 'resolved', resolution_side = 'yes' WHERE id = ?", (market_id,))
    db.commit()

    print("\nDistributing Winnings...")
    distribute_winnings(db, market_id)

    print("\nFinal Balances (Notice the payout difference for identical shares due to time weighting):")
    final_data = []
    for uid in user_ids:
        b = cursor.execute("SELECT balance FROM users WHERE id = ?", (uid,)).fetchone()['balance']
        net = b - 1000.0
        final_data.append({"User": uid, "Final Balance": b, "Net Profit": net})
        
    df = pd.DataFrame(final_data)
    print(df.to_string(index=False))
    
    # Cleanup
    cursor.execute("DELETE FROM bets WHERE market_id = ?", (market_id,))
    cursor.execute("DELETE FROM markets WHERE id = ?", (market_id,))
    for uid in user_ids:
        cursor.execute("DELETE FROM users WHERE id = ?", (uid,))
    db.commit()
    db.close()

run_multi_user_weight_test()

/var/folders/s9/7x6wjtq126773lhfsl5rqqv40000gn/T/ipykernel_61517/3109330841.py:15: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO users (id, username, balance, created_at) VALUES (?, ?, ?, ?)",
/var/folders/s9/7x6wjtq126773lhfsl5rqqv40000gn/T/ipykernel_61517/3109330841.py:22: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO markets (id, name, started_at, ended_at, estimated_resolution_time) VALUES (?, ?, ?, ?, ?)",
2026-04-20 23:56:18,672 INFO User test_user_1 executed trade of 100 shares at price 0.5 on side yes for market test_market_weight_123 at time 2026-04-20 21:42:58.671439 with weight 2.979144363545937
2026-04-20 23:56:18,673 INFO User test_user_2 executed trade of 100 shares at price 0.5 on side yes for market test_market_wei

Initial Balances:
test_user_1: 1000.0
test_user_2: 1000.0
test_user_3: 1000.0
------------------------------

Distributing Winnings...

Final Balances (Notice the payout difference for identical shares due to time weighting):
       User  Final Balance  Net Profit
test_user_1    1051.271226   51.271226
test_user_2    1048.728774   48.728774
test_user_3     900.000000 -100.000000
